In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor 
import xgboost as xgb
from warnings import filterwarnings
filterwarnings("ignore")
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv")

# check shape from dataset
print(train.shape, test.shape)

(1460, 81) (1459, 80)


In [3]:
train_id = train["Id"]
test_id = test["Id"]

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [5]:
train.describe()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
count,1460.000000,1460.000000,1201.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1452.000000,1460.000000,...,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,730.500000,56.897260,70.049958,10516.828082,6.099315,5.575342,1971.267808,1984.865753,103.685262,443.639726,...,94.244521,46.660274,21.954110,3.409589,15.060959,2.758904,43.489041,6.321918,2007.815753,180921.195890
std,421.610009,42.300571,24.284752,9981.264932,1.382997,1.112799,30.202904,20.645407,181.066207,456.098091,...,125.338794,66.256028,61.119149,29.317331,55.757415,40.177307,496.123024,2.703626,1.328095,79442.502883
min,1.000000,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000
25%,365.750000,20.000000,59.000000,7553.500000,5.000000,5.000000,1954.000000,1967.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,129975.000000
50%,730.500000,50.000000,69.000000,9478.500000,6.000000,5.000000,1973.000000,1994.000000,0.000000,383.500000,...,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,163000.000000
75%,1095.250000,70.000000,80.000000,11601.500000,7.000000,6.000000,2000.000000,2004.000000,166.000000,712.250000,...,168.000000,68.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,214000.000000
max,1460.000000,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,...,857.000000,547.000000,552.000000,508.000000,480.000000,738.000000,15500.000000,12.000000,2010.000000,755000.000000


In [6]:
train.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [7]:
full_data = pd.concat([train, test], axis=0).reset_index(drop=True)
print("Combined shape:", full_data.shape)

Combined shape: (2919, 81)


In [8]:
num_cols = full_data.select_dtypes(include=['int64','float64']).columns
for col in num_cols:
    full_data[col] = full_data[col].fillna(full_data[col].median())
print(full_data[col])

0       208500.0
1       181500.0
2       223500.0
3       140000.0
4       250000.0
          ...   
2914    163000.0
2915    163000.0
2916    163000.0
2917    163000.0
2918    163000.0
Name: SalePrice, Length: 2919, dtype: float64


In [9]:
cat_cols = full_data.select_dtypes(include=['object']).columns
for col in cat_cols:
    full_data[col] = full_data[col].fillna(full_data[col].mode()[0])

print("Missing values after fill:", full_data.isnull().sum().sum())


Missing values after fill: 0


In [10]:
le = LabelEncoder()
for col in cat_cols:
    full_data[col] = le.fit_transform(full_data[col].astype(str))
print(full_data[col])

0       4
1       4
2       4
3       0
4       4
       ..
2914    4
2915    0
2916    0
2917    4
2918    4
Name: SaleCondition, Length: 2919, dtype: int64


In [11]:
full_data['TotalSF'] = (full_data['TotalBsmtSF'] +
                        full_data['1stFlrSF'] +
                        full_data['2ndFlrSF'])

# Total bathrooms
full_data['TotalBath'] = (full_data['FullBath'] +
                          full_data['HalfBath'] * 0.5 +
                          full_data['BsmtFullBath'] +
                          full_data['BsmtHalfBath'] * 0.5)

# House age
full_data['HouseAge']    = full_data['YrSold'] - full_data['YearBuilt']
full_data['RemodAge']    = full_data['YrSold'] - full_data['YearRemodAdd']

# Has garage?
full_data['HasGarage']   = (full_data['GarageArea'] > 0).astype(int)

# Has pool?
full_data['HasPool']     = (full_data['PoolArea'] > 0).astype(int)

# Has basement?
full_data['HasBasement'] = (full_data['TotalBsmtSF'] > 0).astype(int)
print("Features after engineering:", full_data.shape[1])



Features after engineering: 88


In [12]:

# Separate target
y = np.log1p(train["SalePrice"])
X = train.drop(["SalePrice"], axis=1)

# Fill missing
X = X.fillna(0)
test = test.fillna(0)

# Convert categorical to numeric
X = pd.get_dummies(X)
test = pd.get_dummies(test)

# Align columns
X, test = X.align(test, join='left', axis=1, fill_value=0)

# Train
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X, y)

# Predict
preds = np.expm1(model.predict(test))
preds


array([125091.44498009, 153633.18709772, 177178.38172318, ...,
       150614.85065225, 110498.83052953, 234127.74249959])

In [13]:
X = pd.get_dummies(X)
test = pd.get_dummies(test)

# Align columns
X, test = X.align(test, join='left', axis=1, fill_value=0)


In [14]:
import xgboost as xgb

print("Training XGBoost")

model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.15,
    reg_lambda=0.8,
    random_state=42,
    n_jobs=-1
)

# CV Score Check

# Final Training
model.fit(X, y)

# Prediction
pred_log = model.predict(X)
pred = np.expm1(pred_log)


Training XGBoost


In [15]:
# Submission
submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": preds
})

submission.to_csv("submission.csv", index=False)

print("Submission ready ✅")

Submission ready ✅
